In [1]:
#testing
from reachy_sdk import ReachySDK
reachy = ReachySDK(host='localhost')

# # Inspect what camera attributes exist
# print(dir(reachy))

# # Try these — one should work for 2021 Reachy
# print(type(reachy.left_camera))
# print(dir(reachy.left_camera))

# # Get a frame
# frame = reachy.left_camera.last_frame
# print(frame.shape)  # Should be (H, W, 3) numpy array

In [ ]:
#YOLO
from ultralytics import YOLO
model = YOLO("yolov8n.pt")   # downloads ~6MB on first run automatically

def detect_objects(frame):
    results = model(frame, verbose=False)[0]
    return results.plot()    # returns frame with boxes+labels drawn

from reachy_sdk import ReachySDK
from http.server import BaseHTTPRequestHandler, HTTPServer
import cv2
import numpy as np
import threading
import urllib.request
import os

# ── Download model weights if not present ──────────────────────────────────────
MODEL_DIR = "model"
os.makedirs(MODEL_DIR, exist_ok=True)

PROTO   = f"{MODEL_DIR}/MobileNetSSD_deploy.prototxt"
WEIGHTS = f"{MODEL_DIR}/MobileNetSSD_deploy.caffemodel"

if not os.path.exists(PROTO):
    print("Downloading prototxt...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/chuanqi305/MobileNet-SSD/master/MobileNetSSD_deploy.prototxt",
        PROTO
    )
if not os.path.exists(WEIGHTS):
    print("Downloading weights (~23MB)...")
    urllib.request.urlretrieve(
        "https://drive.google.com/uc?export=download&id=0B3gersZ2cHIxRm5PMWRoTkdHdHc",
        WEIGHTS
    )

CLASSES = [
    "background", "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow", "diningtable", "dog",
    "horse", "motorbike", "person", "pottedplant", "sheep",
    "sofa", "train", "tvmonitor"
]

# Distinct color per class
COLORS = np.random.default_rng(42).integers(100, 255, size=(len(CLASSES), 3)).tolist()

net = cv2.dnn.readNetFromCaffe(PROTO, WEIGHTS)
print("Model loaded.")

# ── Reachy init ────────────────────────────────────────────────────────────────
reachy = ReachySDK(host='localhost')
reachy.left_camera.wait_for_new_frame()
reachy.right_camera.wait_for_new_frame()
print("Cameras ready, starting stream...")


# ── Frame builders ─────────────────────────────────────────────────────────────
def get_detection_frame():
    reachy.left_camera.wait_for_new_frame()
    frame = reachy.left_camera.last_frame.copy()
    frame = detect_objects(frame)
    cv2.putText(frame, "OBJECT DETECTION",
                (10, frame.shape[0] - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 136), 1)
    return frame

def get_stereo_detection_frame():
    """Side-by-side with detection on both cameras."""
    reachy.left_camera.wait_for_new_frame()
    reachy.right_camera.wait_for_new_frame()
    left  = detect_objects(reachy.left_camera.last_frame.copy())
    right = detect_objects(reachy.right_camera.last_frame.copy())
    cv2.putText(left,  'LEFT',  (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 100), 2)
    cv2.putText(right, 'RIGHT', (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 180, 255), 2)
    return np.hstack([left, right])

# ── HTTP server ────────────────────────────────────────────────────────────────
class MJPEGHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path in ('/stream', '/stream/stereo'):
            self.send_response(200)
            self.send_header('Content-type', 'multipart/x-mixed-replace; boundary=frame')
            self.end_headers()
            grab = get_stereo_detection_frame 
#             if 'stereo' in self.path else get_detection_frame
            try:
                while True:
                    frame = grab()
                    _, jpg = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 80])
                    self.wfile.write(b'--frame\r\n')
                    self.wfile.write(b'Content-Type: image/jpeg\r\n\r\n')
                    self.wfile.write(jpg.tobytes())
                    self.wfile.write(b'\r\n')
            except (BrokenPipeError, ConnectionResetError):
                pass

        elif self.path == '/':
            self.send_response(200)
            self.send_header('Content-type', 'text/html')
            self.end_headers()
            self.wfile.write(b"""
            <html><head><title>Reachy Detection</title>
            <style>
                body  { background:#111; color:#eee; font-family:sans-serif;
                        display:flex; flex-direction:column;
                        align-items:center; padding:20px; margin:0; }
                h1    { color:#00ff88; margin-bottom:12px; }
                nav   { margin-bottom:16px; }
                nav a { color:#00ccff; margin:0 12px; text-decoration:none;
                        font-size:1.1em; }
                img   { border:2px solid #00ff88; max-width:100%; }
            </style></head>
            <body>
                <h1>Reachy Object Detection</h1>
                
                <img src="/stream" id="feed"
                     onerror="setTimeout(()=>location.reload(),1000)">
                <script>
                  const p = new URLSearchParams(location.search);
                  if (p.get('view') === 'stereo')
                      document.getElementById('feed').src = '/stream/stereo';
                </script>
            </body></html>""")
        else:
            self.send_response(404)
            self.end_headers()

    def log_message(self, *args):
        pass

server = HTTPServer(('0.0.0.0', 8080), MJPEGHandler)
print("Stream live at http://localhost:8080")
threading.Thread(target=server.serve_forever, daemon=True).start()
input("Press Enter to stop...\n")

In [ ]:
from pycoral.utils.edgetpu import make_interpreter
from pycoral.adapters import common, detect
from reachy_sdk import ReachySDK
from http.server import BaseHTTPRequestHandler, HTTPServer
import cv2
import numpy as np
import threading
import time

# ── Setup ─────────────────────────────────────────────────────────────────────
MODEL_PATH = '/home/reachy/dev/pycoral/test_data/efficientdet_lite0_320_ptq_edgetpu.tflite'
LABELS_PATH = 'coco_labels.txt'
SCORE_THRESHOLD = 0.4

reachy = ReachySDK(host='localhost')
reachy.left_camera.wait_for_new_frame()

interpreter = make_interpreter(MODEL_PATH)
interpreter.allocate_tensors()
input_size =d common.input_size(interpreter)  # (384, 384)

with open(LABELS_PATH) as f:
    labels = f.read().splitlines()

# COCO classes to highlight as first-aid relevant
FIRST_AID_IDS = {
    76: (0, 255, 100),   # scissors  — green
    39: (0, 180, 255),   # bottle    — yellow
    84: (255, 100, 0),   # book      — blue
}
DEFAULT_COLOR = (180, 180, 180)  # gray for everything else

# ── Shared state ──────────────────────────────────────────────────────────────
latest_frame  = None
latest_boxes  = []
frame_lock    = threading.Lock()
boxes_lock    = threading.Lock()
fps_display   = {'stream': 0, 'detect': 0}

# ── Thread 1: capture frames at full camera rate ──────────────────────────────
def capture_loop():
    global latest_frame
    t_last = time.time()
    count  = 0
    while True:
        reachy.left_camera.wait_for_new_frame()
        with frame_lock:
            latest_frame = reachy.left_camera.last_frame.copy()
        count += 1
        if time.time() - t_last >= 1.0:
            fps_display['stream'] = count
            count  = 0
            t_last = time.time()

threading.Thread(target=capture_loop, daemon=True).start()

# ── Thread 2: run EfficientDet on Coral as fast as possible ──────────────────
def detection_loop():
    global latest_boxes
    while True:
        # Always grab the LATEST frame, skip any queued ones
        with frame_lock:
            frame = latest_frame
        if frame is None:
            time.sleep(0.01)
            continue

        rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(rgb, input_size)
        common.set_input(interpreter, resized)
        interpreter.invoke()

        objs  = detect.get_objects(interpreter, score_threshold=SCORE_THRESHOLD)
        h, w  = frame.shape[:2]
        sx, sy = w / input_size[0], h / input_size[1]
        boxes = []
        for obj in objs:
            x1 = int(obj.bbox.xmin * sx)
            y1 = int(obj.bbox.ymin * sy)
            x2 = int(obj.bbox.xmax * sx)
            y2 = int(obj.bbox.ymax * sy)
            label = labels[obj.id] if obj.id < len(labels) else '?'
            color = FIRST_AID_IDS.get(obj.id, DEFAULT_COLOR)
            boxes.append((x1, y1, x2, y2, label, obj.score, color, obj.id in FIRST_AID_IDS))

        with boxes_lock:
            latest_boxes = boxes
        # No sleep — immediately grab the next latest frame
        # This ensures detection always works on current reality, not queued frames

# ── Composite: draw boxes on latest frame ─────────────────────────────────────
def get_annotated_frame():
    with frame_lock:
        frame = latest_frame
    if frame is None:
        return None
    frame = frame.copy()

    with boxes_lock:
        boxes = list(latest_boxes)

    for (x1, y1, x2, y2, label, score, color, is_fa) in boxes:
        thickness = 2 if is_fa else 1
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)

        # Label background for readability
        text     = f'{label} {score:.0%}'
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(frame, (x1, y1 - th - 8), (x1 + tw + 4, y1), color, -1)
        cv2.putText(frame, text, (x1 + 2, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

    # HUD: stream and detection FPS
    cv2.putText(frame,
                f"Stream: {fps_display['stream']} FPS  |  Detect: {fps_display['detect']} FPS",
                (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return frame

# ── MJPEG server ──────────────────────────────────────────────────────────────
class MJPEGHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == '/stream':
            self.send_response(200)
            self.send_header('Content-type', 'multipart/x-mixed-replace; boundary=frame')
            self.end_headers()
            try:
                while True:
                    frame = get_annotated_frame()
                    if frame is None:
                        time.sleep(0.01)
                        continue
                    _, jpg = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 75])
                    self.wfile.write(b'--frame\r\n')
                    self.wfile.write(b'Content-Type: image/jpeg\r\n\r\n')
                    self.wfile.write(jpg.tobytes())
                    self.wfile.write(b'\r\n')
            except (BrokenPipeError, ConnectionResetError):
                pass

        elif self.path == '/':
            self.send_response(200)
            self.send_header('Content-type', 'text/html')
            self.end_headers()
            html = """<html><head><title>Reachy Vision</title>
            <style>
              body { background:#111; color:#eee; font-family:monospace; text-align:center; }
              img  { border:2px solid #00ff88; max-width:100%; margin-top:12px; }
            </style></head>
            <body>
              <h3>Reachy EfficientDet - Live Detection</h3>
              <img src="/stream" onerror="setTimeout(()=>location.reload(),1000)">
              
            </body></html>"""
            self.wfile.write(html.encode('utf-8'))
        else:
            self.send_response(404)
            self.end_headers()

    def log_message(self, format, *args):
        pass

# Shut down any previous server instance cleanly
try:
    server.shutdown()
except:
    pass

server = HTTPServer(('0.0.0.0', 8080), MJPEGHandler)
print("Live detection stream at http://localhost:8080")
threading.Thread(target=server.serve_forever, daemon=True).start()

In [1]:
#dept overlay
from reachy_sdk import ReachySDK
from http.server import BaseHTTPRequestHandler, HTTPServer
import cv2
import numpy as np  # ← was missing
import threading

reachy = ReachySDK(host='localhost')

reachy.left_camera.wait_for_new_frame()
reachy.right_camera.wait_for_new_frame()
print("Cameras ready, starting stream...")

#single frame with depth overlay
def get_depth_overlay_frame():
    reachy.left_camera.wait_for_new_frame()
    reachy.right_camera.wait_for_new_frame()
    left  = reachy.left_camera.last_frame.copy()
    right = reachy.right_camera.last_frame.copy()

    gray_l = cv2.cvtColor(left,  cv2.COLOR_BGR2GRAY)
    gray_r = cv2.cvtColor(right, cv2.COLOR_BGR2GRAY)

    # Stereo block matching — no calibration needed for a rough depth map
    stereo = cv2.StereoBM_create(numDisparities=64, blockSize=15)
    disparity = stereo.compute(gray_l, gray_r)

    # Normalize disparity to 0-255 for visualization
    disp_norm = cv2.normalize(disparity, None, 0, 255, cv2.NORM_MINMAX)
    disp_norm = np.uint8(disp_norm)

    # Colorize: red = close, blue = far (COLORMAP_JET)
    depth_color = cv2.applyColorMap(disp_norm, cv2.COLORMAP_JET)

    # Blend onto left camera image: 60% real image, 40% depth color
    combined = cv2.addWeighted(left, 0.6, depth_color, 0.4, 0)

    cv2.putText(combined, 'DEPTH OVERLAY',
                (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
    return combined  # single 640x480 frame


#both frames side by side
def get_stereo_frame():
    reachy.left_camera.wait_for_new_frame()
    reachy.right_camera.wait_for_new_frame()
    left  = reachy.left_camera.last_frame.copy()
    right = reachy.right_camera.last_frame.copy()
    cv2.putText(left,  'LEFT',  (10,20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,100), 2)
    cv2.putText(right, 'RIGHT', (10,20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,180,255), 2)
    return np.hstack([left, right])  # 1280x480

class MJPEGHandler(BaseHTTPRequestHandler):

    def do_GET(self):
        if self.path == '/stream':
            self.send_response(200)
            self.send_header('Content-type', 'multipart/x-mixed-replace; boundary=frame')
            self.end_headers()
            try:
                while True:
                    frame = get_depth_overlay_frame()  # ← call it as a function with ()
                    _, jpg = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 75])
                    self.wfile.write(b'--frame\r\n')
                    self.wfile.write(b'Content-Type: image/jpeg\r\n\r\n')
                    self.wfile.write(jpg.tobytes())
                    self.wfile.write(b'\r\n')
            except (BrokenPipeError, ConnectionResetError):
                pass

        elif self.path == '/':
            self.send_response(200)
            self.send_header('Content-type', 'text/html')
            self.end_headers()
            self.wfile.write(b"""
            <html><head><title>Reachy Camera</title>
            <style>
                body { background:#111; display:flex; justify-content:center;
                       align-items:center; height:100vh; margin:0; }
                img  { border: 2px solid #00ff88; max-width:100%; }
            </style></head>
            <body><img src="/stream" onerror="setTimeout(()=>location.reload(),1000)"></body>
            </html>""")
        else:
            self.send_response(404)
            self.end_headers()

    def log_message(self, format, *args):
        pass

server = HTTPServer(('0.0.0.0', 8080), MJPEGHandler)
print("Stream live at http://localhost:8080")
threading.Thread(target=server.serve_forever, daemon=True).start()

Cameras ready, starting stream...
Stream live at http://localhost:8080


In [2]:
from pycoral.utils.edgetpu import make_interpreter
from pycoral.adapters import common
import numpy as np
import time

# Cell 2: Quick benchmark of Lite0 (faster, less accurate)
MODEL_LITE0 = '/home/reachy/dev/pycoral/test_data/efficientdet_lite0_320_ptq_edgetpu.tflite'

interp0 = make_interpreter(MODEL_LITE0)
interp0.allocate_tensors()
sz0 = common.input_size(interp0)

dummy = np.zeros((sz0[1], sz0[0], 3), dtype=np.uint8)
common.set_input(interp0, dummy)
interp0.invoke()  # warm up

start = time.time()
for _ in range(100):
    common.set_input(interp0, dummy)
    interp0.invoke()
elapsed = time.time() - start
print(f"Lite0 ({sz0}): {100/elapsed:.1f} FPS")

Lite0 ((320, 320)): 19.8 FPS


In [ ]:
from pycoral.adapters import detect
import cv2
from reachy_sdk import ReachySDK
from http.server import BaseHTTPRequestHandler, HTTPServer
import cv2
import numpy as np  # ← was missing
import threading

reachy = ReachySDK(host='localhost')

# Load labels — already in your working directory
with open('coco_labels.txt') as f:
    labels = f.read().splitlines()

reachy.left_camera.wait_for_new_frame()
frame = reachy.left_camera.last_frame.copy()

# Resize to model input size
rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
resized = cv2.resize(rgb, input_size)

common.set_input(interpreter, resized)
interpreter.invoke()

objects = detect.get_objects(interpreter, score_threshold=0.4)

print(f"Detected {len(objects)} objects:")
for obj in objects:
    print(f"  {labels[obj.id]}  —  {obj.score:.0%}  —  bbox: {obj.bbox}")

Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 154, in grpc._cython.cygrpc.PollerCompletionQueue._handle_events
BlockingIOError: [Errno 11] Resource temporarily unavailable
Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 154, in grpc._cython.cygrpc.PollerCompletion

Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 154, in grpc._cython.cygrpc.PollerCompletionQueue._handle_events
BlockingIOError: [Errno 11] Resource temporarily unavailable
Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 154, in grpc._cython.cygrpc.PollerCompletion

Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 154, in grpc._cython.cygrpc.PollerCompletionQueue._handle_events
BlockingIOError: [Errno 11] Resource temporarily unavailable
Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 154, in grpc._cython.cygrpc.PollerCompletion

Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 154, in grpc._cython.cygrpc.PollerCompletionQueue._handle_events
BlockingIOError: [Errno 11] Resource temporarily unavailable
Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 154, in grpc._cython.cygrpc.PollerCompletion

Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 154, in grpc._cython.cygrpc.PollerCompletionQueue._handle_events
BlockingIOError: [Errno 11] Resource temporarily unavailable
Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 154, in grpc._cython.cygrpc.PollerCompletion

...

Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 154, in grpc._cython.cygrpc.PollerCompletionQueue._handle_events
BlockingIOError: [Errno 11] Resource temporarily unavailable
Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 154, in grpc._cython.cygrpc.PollerCompletion

Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 154, in grpc._cython.cygrpc.PollerCompletionQueue._handle_events
BlockingIOError: [Errno 11] Resource temporarily unavailable
Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 154, in grpc._cython.cygrpc.PollerCompletion

Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 154, in grpc._cython.cygrpc.PollerCompletionQueue._handle_events
BlockingIOError: [Errno 11] Resource temporarily unavailable
Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 154, in grpc._cython.cygrpc.PollerCompletion

Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 154, in grpc._cython.cygrpc.PollerCompletionQueue._handle_events
BlockingIOError: [Errno 11] Resource temporarily unavailable
Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 154, in grpc._cython.cygrpc.PollerCompletion

Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 154, in grpc._cython.cygrpc.PollerCompletionQueue._handle_events
BlockingIOError: [Errno 11] Resource temporarily unavailable
Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 154, in grpc._cython.cygrpc.PollerCompletion